# News Topic Classifier Using BERT
This notebook demonstrates fine-tuning a BERT model on the AG News dataset.

In [11]:
!pip install -q datasets transformers[torch] evaluate

In [12]:
from datasets import load_dataset

# Load the AG News dataset
dataset = load_dataset('ag_news')
print(dataset)

# Display a sample entry
print(f"\nSample headline: {dataset['train'][0]['text']}")
print(f"Label: {dataset['train'][0]['label']}")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Sample headline: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Label: 2


In [13]:
from transformers import AutoTokenizer

# Initialize the tokenizer
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Shuffle and take a small subset of the dataset for speed
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(200))

# Tokenize the subsets
tokenized_train = small_train_dataset.map(preprocess_function, batched=True)
tokenized_test = small_test_dataset.map(preprocess_function, batched=True)

# Rename and format
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

tokenized_test = tokenized_test.rename_column("label", "labels")
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Tokenization of small subset complete.")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenization of small subset complete.


In [14]:
import evaluate
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

In [15]:
# Load the model with 4 labels
num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

print(f"Model '{model_checkpoint}' loaded for {num_labels} classes.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model 'bert-base-uncased' loaded for 4 classes.


In [16]:
training_args = TrainingArguments(
    output_dir="./bert-news-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer initialized with small subset.")

Trainer initialized with small subset.


In [18]:
# Start training on the small subset
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.467820,0.544282,0.825000,0.824727


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=63, training_loss=0.6135443467942495, metrics={'train_runtime': 1326.0563, 'train_samples_per_second': 0.754, 'train_steps_per_second': 0.048, 'total_flos': 65778945024000.0, 'train_loss': 0.6135443467942495, 'epoch': 1.0})

In [20]:
metrics = trainer.evaluate()
print(metrics)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.5442823171615601, 'eval_accuracy': 0.825, 'eval_f1': 0.8247269123904638, 'eval_runtime': 103.5419, 'eval_samples_per_second': 1.932, 'eval_steps_per_second': 0.126, 'epoch': 1.0}


### Model Evaluation Results

Based on the evaluation, the model achieved the following metrics:

*   **Loss**: Measures the error of the model. A lower loss is better.
*   **Accuracy**: The proportion of correctly classified instances. A higher accuracy is better.
*   **F1 Score**: The harmonic mean of precision and recall. It's a good metric for imbalanced datasets. A higher F1 score is better.

In [21]:
print(f"Evaluation Loss: {metrics['eval_loss']:.4f}")
print(f"Evaluation Accuracy: {metrics['eval_accuracy']:.4f}")
print(f"Evaluation F1 Score (weighted): {metrics['eval_f1']:.4f}")

Evaluation Loss: 0.5443
Evaluation Accuracy: 0.8250
Evaluation F1 Score (weighted): 0.8247


Next, we can consider saving the fine-tuned model or visualizing the results further.